# Notebook 12: Model Stealing — Extracting a Black-Box Model

## Why This Matters

When a company deploys an ML model as an API, they assume their intellectual property is safe behind that interface. The training data cost millions to collect. The model architecture took months to tune. The API charges per query. Model stealing attacks challenge all of these assumptions — an adversary with only black-box API access can often reconstruct a functionally equivalent model at a fraction of the original cost.

This is not academic. In 2016, researchers stole production models from BigML and Amazon ML using nothing but their public APIs. In 2020, OpenAI began aggressively rate-limiting GPT-3 after Tramer et al. demonstrated systematic extraction. In 2022, Microsoft Azure ML models were shown to be extractable with under \$10 in API costs.

Model stealing sits at the intersection of IP law, competitive intelligence, and ML security — and it has no clean solution.

## Background

### The Taxonomy of Extraction Attacks

Tramer et al.'s 2016 paper *"Stealing Machine Learning Models via Prediction APIs"* established the first systematic framework. They identified three levels of adversary capability:

- **Soft-label access**: The API returns full probability distributions over classes. This is the richest signal — the distribution encodes far more information than a hard label.
- **Hard-label access**: The API returns only the predicted class. Harder, but still possible with enough queries.
- **Confidence scores**: Some APIs return a confidence percentage. Intermediate signal.

The attack goal is equally variable: the adversary might want a model that mimics the *decision boundary* (functionally equivalent), or one that mimics the *input-output behavior* on a specific distribution, or one that reveals the original *architecture/hyperparameters*.

### Knockoff Nets

Orekondy et al.'s 2019 *Knockoff Nets* paper made extraction practical. The key insight: you don't need data from the same distribution as the victim's training set. You just need a dataset whose inputs cover the feature space well enough that querying them teaches the surrogate the victim's decision boundaries.

Their algorithm:
1. Sample queries from a *transfer set* (CIFAR, ImageNet, or even random images)
2. Query the victim API for soft labels on each
3. Train a surrogate model on (query, soft-label) pairs using knowledge distillation loss
4. Repeat

The surrogate doesn't need to look like the victim internally — it just needs to agree on outputs. On ImageNet classifiers, Knockoff Nets achieved 95%+ agreement with the victim using only 50k queries (vs. 1.2M images in ImageNet).

### Why Soft Labels Are So Rich

Hinton et al.'s knowledge distillation work (2015) showed that class probabilities carry *dark knowledge* — information about inter-class similarities that hard labels discard. A model trained on MNIST soft labels like `[0.001, 0.92, 0.03, 0.04, ...]` learns that a slightly ambiguous "2" looks a bit like a "7", which a hard label of `2` never conveys. This is why surrogate models trained on soft labels often generalize *better* than models trained on the same hard-labeled data from the same distribution.

### Active Learning for Efficient Stealing

If API queries cost money, an adversary wants maximum information per query. Jacobsen et al. (2018) proposed *adaptive* extraction: instead of sampling queries randomly, use the current surrogate model's uncertainty to pick the most informative next query. This is exactly active learning — exploiting the surrogate's own uncertainty map to guide where to probe the victim. Empirically, adaptive strategies achieve the same surrogate accuracy with 2-5x fewer queries.

### Defenses and Their Limitations

**Rate limiting** is the most common defense — cap queries per IP/API key. But adversaries can buy API access from many accounts, distribute queries across IP ranges, or simply operate slowly over weeks.

**Output perturbation** adds noise to the returned probabilities, degrading the signal. But enough queries can average out Gaussian noise, and heavy noise degrades legitimate users too.

**Watermarking** (covered in notebook 17) embeds a detectable pattern in the model's outputs or weights. If a suspected knockoff model produces the watermark when queried with trigger inputs, you have forensic evidence. But watermarks can sometimes be removed through fine-tuning.

**Query detection** builds a classifier to flag extraction-pattern queries — they often have different distributional properties than normal user queries (uniform sampling vs. naturalistic distribution). This works in principle but requires knowing what "normal" looks like.

The uncomfortable truth: if your model is deployed as a public API and an adversary can afford enough queries, extraction is probably unavoidable. The question is whether you can detect it and whether the cost of extraction exceeds the economic value.

## Structure of This Notebook

1. Train a victim MNIST classifier (the "proprietary model")
2. Implement Knockoff Nets extraction with soft-label distillation
3. Compare hard-label vs. soft-label extraction efficiency
4. Implement adaptive (uncertainty-based) query selection
5. Implement rate-limiting and output-perturbation defenses
6. Measure attack accuracy vs. query budget tradeoff

## What You'll Learn

- Why soft labels carry more information than hard labels (dark knowledge)
- How knowledge distillation loss enables surrogate training without the original data
- The query budget vs. accuracy tradeoff for extraction attacks
- How rate limiting and output perturbation degrade extraction
- Why watermarking is the most forensically useful defense

In [ ]:
%pip install torch torchvision numpy matplotlib scikit-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

device = (
    torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cuda') if torch.cuda.is_available()
    else torch.device('cpu')
)
print(f'Device: {device}')

torch.manual_seed(42)
np.random.seed(42)

## 1. Victim Model — The "Proprietary API"

In [ ]:
class MnistCNN(nn.Module):
    """Standard victim model — represents a deployed proprietary classifier."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


class VictimAPI:
    """
    Simulates a black-box ML API. Adversary can only call query().
    Internally enforces rate limits and output perturbation.
    """
    def __init__(self, model, rate_limit=None, noise_std=0.0):
        self.model = model
        self.rate_limit = rate_limit          # max queries per call batch
        self.noise_std = noise_std            # Gaussian noise on logits
        self.query_count = 0
        self.model.eval()

    def query(self, x, return_hard_label=False):
        """Query the black-box API. Returns soft labels (probabilities) by default."""
        if self.rate_limit and len(x) > self.rate_limit:
            raise ValueError(f'Rate limit exceeded: max {self.rate_limit} per request')

        self.query_count += len(x)

        with torch.no_grad():
            x = x.to(device)
            logits = self.model(x)

            if self.noise_std > 0:
                logits = logits + torch.randn_like(logits) * self.noise_std

            probs = F.softmax(logits, dim=-1)

        if return_hard_label:
            return probs.argmax(dim=1).cpu()
        return probs.cpu()

    def reset_counter(self):
        self.query_count = 0


# Load MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Train victim model
victim_model = MnistCNN().to(device)
optimizer = torch.optim.Adam(victim_model.parameters(), lr=1e-3)

victim_model.train()
for epoch in range(3):
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(victim_model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}')

# Evaluate victim
victim_model.eval()
correct = sum(
    (victim_model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    for x, y in test_loader
)
victim_acc = correct / len(test_dataset)
print(f'\nVictim model test accuracy: {victim_acc:.4f}')

# Create the black-box API
api = VictimAPI(victim_model)
print('Black-box API created. Adversary has no access to weights or architecture.')

## 2. Knockoff Nets — Soft-Label Extraction

The adversary samples from a *transfer set* (here: a disjoint subset of MNIST training data, mimicking a different-distribution dataset), queries the victim API for soft labels, then trains a surrogate via knowledge distillation.

In [ ]:
class SurrogateCNN(nn.Module):
    """Surrogate (knockoff) model. Architecture can differ from victim."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def build_transfer_set(dataset, n_samples):
    """Sample a transfer set (adversary's query inputs — no labels needed)."""
    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    imgs = torch.stack([dataset[i][0] for i in indices])
    return imgs


def knockoff_extraction(
    api: VictimAPI,
    transfer_images: torch.Tensor,
    n_epochs: int = 5,
    temperature: float = 4.0,
    batch_size: int = 256,
    use_soft_labels: bool = True,
):
    """
    Knockoff Nets extraction.

    Step 1: Query victim API for labels on transfer set.
    Step 2: Train surrogate via knowledge distillation (soft) or CE (hard).

    Temperature T > 1 softens distributions further, amplifying dark knowledge.
    """
    # --- Query phase ---
    all_probs = []
    for i in range(0, len(transfer_images), batch_size):
        batch = transfer_images[i:i+batch_size]
        if use_soft_labels:
            probs = api.query(batch, return_hard_label=False)
        else:
            labels = api.query(batch, return_hard_label=True)
            probs = F.one_hot(labels, num_classes=10).float()
        all_probs.append(probs)

    soft_labels = torch.cat(all_probs, dim=0)   # (N, 10)
    print(f'Queried {len(transfer_images)} samples. Total API calls: {api.query_count}')

    # --- Training phase ---
    surrogate = SurrogateCNN().to(device)
    opt = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
    steal_dataset = TensorDataset(transfer_images, soft_labels)
    steal_loader = DataLoader(steal_dataset, batch_size=batch_size, shuffle=True)

    T = temperature

    for epoch in range(n_epochs):
        surrogate.train()
        total_loss = 0
        for x_batch, y_soft in steal_loader:
            x_batch = x_batch.to(device)
            y_soft = y_soft.to(device)
            opt.zero_grad()
            logits = surrogate(x_batch)

            if use_soft_labels:
                # Knowledge distillation loss: KL(soft_labels || surrogate_probs)
                # Temperature scaling: divide logits by T before softmax
                log_probs = F.log_softmax(logits / T, dim=-1)
                soft_targets = (y_soft + 1e-8).log()  # avoid log(0)
                # Use KL: sum(p * log(p/q)) = sum(p*log(p)) - sum(p*log(q))
                loss = F.kl_div(log_probs, y_soft, reduction='batchmean') * (T ** 2)
            else:
                # Hard label: standard cross-entropy against one-hot
                hard_labels = y_soft.argmax(dim=1)
                loss = F.cross_entropy(logits, hard_labels)

            loss.backward()
            opt.step()
            total_loss += loss.item()

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}: distillation_loss={total_loss/len(steal_loader):.4f}')

    return surrogate


def evaluate_surrogate(surrogate, test_loader, victim_model):
    """Measure surrogate test accuracy AND agreement with victim."""
    surrogate.eval()
    victim_model.eval()
    correct_clean = 0
    agree = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            surr_pred = surrogate(x).argmax(1)
            vict_pred = victim_model(x).argmax(1)
            correct_clean += (surr_pred == y).sum().item()
            agree += (surr_pred == vict_pred).sum().item()
            total += len(y)

    acc = correct_clean / total
    agreement = agree / total
    return acc, agreement


# Build transfer set from first 10k training images (no true labels used)
transfer_images = build_transfer_set(train_dataset, n_samples=10_000)
print(f'Transfer set: {len(transfer_images)} images')

# Soft-label extraction
api.reset_counter()
print('\n--- Soft-label (Knockoff Nets) extraction ---')
surrogate_soft = knockoff_extraction(api, transfer_images, n_epochs=5, use_soft_labels=True)
acc_soft, agree_soft = evaluate_surrogate(surrogate_soft, test_loader, victim_model)
queries_soft = api.query_count
print(f'Soft-label surrogate: accuracy={acc_soft:.4f}, agreement={agree_soft:.4f}, queries={queries_soft}')

# Hard-label extraction
api.reset_counter()
print('\n--- Hard-label extraction ---')
surrogate_hard = knockoff_extraction(api, transfer_images, n_epochs=5, use_soft_labels=False)
acc_hard, agree_hard = evaluate_surrogate(surrogate_hard, test_loader, victim_model)
queries_hard = api.query_count
print(f'Hard-label surrogate: accuracy={acc_hard:.4f}, agreement={agree_hard:.4f}, queries={queries_hard}')

print(f'\nVictim accuracy: {victim_acc:.4f}')
print(f'Soft-label extraction captured {agree_soft/victim_acc*100:.1f}% of victim behavior')
print(f'Hard-label extraction captured {agree_hard/victim_acc*100:.1f}% of victim behavior')

## 3. Query Budget vs. Accuracy Tradeoff

Real attackers are budget-constrained. How does surrogate quality degrade with fewer queries?

In [ ]:
query_budgets = [500, 1000, 2000, 5000, 10_000]
soft_agreements = []
hard_agreements = []

for budget in query_budgets:
    subset = transfer_images[:budget]

    # Soft
    api.reset_counter()
    s = knockoff_extraction(api, subset, n_epochs=5, use_soft_labels=True)
    _, ag = evaluate_surrogate(s, test_loader, victim_model)
    soft_agreements.append(ag)

    # Hard
    api.reset_counter()
    s = knockoff_extraction(api, subset, n_epochs=5, use_soft_labels=False)
    _, ag = evaluate_surrogate(s, test_loader, victim_model)
    hard_agreements.append(ag)

    print(f'Budget={budget}: soft={soft_agreements[-1]:.4f}, hard={hard_agreements[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(query_budgets, soft_agreements, 'b-o', label='Soft labels (Knockoff Nets)')
ax.plot(query_budgets, hard_agreements, 'r-o', label='Hard labels only')
ax.axhline(victim_acc, color='gray', linestyle='--', label=f'Victim accuracy ({victim_acc:.3f})')
ax.set_xlabel('API Query Budget')
ax.set_ylabel('Surrogate Agreement with Victim')
ax.set_title('Model Stealing: Query Budget vs. Surrogate Quality')
ax.legend()
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Adaptive Query Selection — Uncertainty Sampling

Instead of sampling randomly, the adversary selects queries where the *current surrogate* is most uncertain. This is active learning applied to model stealing — maximum information gain per API dollar.

In [ ]:
def prediction_entropy(model, images, batch_size=256):
    """Compute entropy of softmax distribution for each image — measure of uncertainty."""
    model.eval()
    entropies = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size].to(device)
            probs = F.softmax(model(batch), dim=-1)
            # Shannon entropy: -sum(p * log(p))
            h = -(probs * (probs + 1e-10).log()).sum(dim=-1)
            entropies.append(h.cpu())
    return torch.cat(entropies)


def adaptive_extraction(
    api: VictimAPI,
    candidate_pool: torch.Tensor,
    initial_budget: int = 500,
    rounds: int = 4,
    queries_per_round: int = 1000,
    n_epochs_per_round: int = 3,
):
    """
    Adaptive model stealing with uncertainty-guided query selection.

    Round 0: random initial seed queries → train surrogate
    Rounds 1-N: pick highest-entropy candidates → query → retrain
    """
    api.reset_counter()

    # Initial random seed
    seed_idx = random.sample(range(len(candidate_pool)), initial_budget)
    queried_imgs = candidate_pool[seed_idx]
    queried_labels = api.query(queried_imgs)
    remaining_pool = candidate_pool[[i for i in range(len(candidate_pool)) if i not in set(seed_idx)]]

    surrogate = SurrogateCNN().to(device)

    query_log = []

    for r in range(rounds + 1):
        # Train surrogate on all queried data so far
        dataset = TensorDataset(queried_imgs, queried_labels)
        loader = DataLoader(dataset, batch_size=128, shuffle=True)
        opt = torch.optim.Adam(surrogate.parameters(), lr=1e-3)

        for _ in range(n_epochs_per_round):
            surrogate.train()
            for x_b, y_b in loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                opt.zero_grad()
                log_p = F.log_softmax(surrogate(x_b) / 4.0, dim=-1)
                loss = F.kl_div(log_p, y_b, reduction='batchmean') * 16
                loss.backward()
                opt.step()

        _, agreement = evaluate_surrogate(surrogate, test_loader, victim_model)
        query_log.append((api.query_count, agreement))
        print(f'Round {r}: queries={api.query_count}, agreement={agreement:.4f}')

        if r < rounds and len(remaining_pool) >= queries_per_round:
            # Select most uncertain candidates
            entropies = prediction_entropy(surrogate, remaining_pool)
            top_idx = entropies.argsort(descending=True)[:queries_per_round]
            new_imgs = remaining_pool[top_idx]
            new_labels = api.query(new_imgs)

            queried_imgs = torch.cat([queried_imgs, new_imgs])
            queried_labels = torch.cat([queried_labels, new_labels])
            keep_mask = torch.ones(len(remaining_pool), dtype=torch.bool)
            keep_mask[top_idx] = False
            remaining_pool = remaining_pool[keep_mask]

    return surrogate, query_log


# Pool: 10k unlabeled transfer images
candidate_pool = transfer_images.clone()

print('--- Adaptive (uncertainty-guided) extraction ---')
surrogate_adaptive, adaptive_log = adaptive_extraction(
    api, candidate_pool,
    initial_budget=500, rounds=4, queries_per_round=1000
)

# Compare random baseline at same query budgets
random_log = []
api.reset_counter()
print('\n--- Random baseline extraction ---')
for budget in [500, 1500, 2500, 3500, 4500]:
    subset = transfer_images[:budget]
    api.reset_counter()
    s = knockoff_extraction(api, subset, n_epochs=5, use_soft_labels=True)
    _, ag = evaluate_surrogate(s, test_loader, victim_model)
    random_log.append((budget, ag))
    print(f'Random budget={budget}: agreement={ag:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

adap_q = [q for q, _ in adaptive_log]
adap_ag = [ag for _, ag in adaptive_log]
rand_q = [q for q, _ in random_log]
rand_ag = [ag for _, ag in random_log]

ax.plot(adap_q, adap_ag, 'b-o', label='Adaptive (uncertainty sampling)')
ax.plot(rand_q, rand_ag, 'r-s', label='Random sampling')
ax.axhline(victim_acc, color='gray', linestyle='--', label=f'Victim accuracy')
ax.set_xlabel('API Queries Used')
ax.set_ylabel('Surrogate Agreement')
ax.set_title('Adaptive vs. Random Query Selection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Defenses: Rate Limiting and Output Perturbation

In [ ]:
def test_defense(noise_std, n_samples=5000, n_epochs=5):
    """Run extraction attack against a defended API (output noise)."""
    defended_api = VictimAPI(victim_model, noise_std=noise_std)
    subset = transfer_images[:n_samples]
    surrogate = knockoff_extraction(defended_api, subset, n_epochs=n_epochs, use_soft_labels=True)
    acc, agreement = evaluate_surrogate(surrogate, test_loader, victim_model)
    return acc, agreement, defended_api.query_count


noise_levels = [0.0, 0.5, 1.0, 2.0, 4.0]
defense_results = []

print('Effect of output perturbation (logit noise) on extraction quality:')
print(f'{"Noise std":>10} {"Surrogate Acc":>15} {"Agreement":>12}')
print('-' * 40)

for noise in noise_levels:
    acc, ag, _ = test_defense(noise, n_samples=5000, n_epochs=5)
    defense_results.append((noise, acc, ag))
    print(f'{noise:>10.1f} {acc:>15.4f} {ag:>12.4f}')

In [ ]:
noises = [r[0] for r in defense_results]
accs = [r[1] for r in defense_results]
ags = [r[2] for r in defense_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(noises, accs, 'b-o')
axes[0].axhline(victim_acc, color='gray', linestyle='--', label='Victim accuracy')
axes[0].set_xlabel('Logit Noise Std')
axes[0].set_ylabel('Surrogate Test Accuracy')
axes[0].set_title('Noise Defense: Accuracy Impact')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(noises, ags, 'r-o')
axes[1].set_xlabel('Logit Noise Std')
axes[1].set_ylabel('Agreement with Victim')
axes[1].set_title('Noise Defense: Agreement Impact')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Output Perturbation Defense vs. Model Stealing')
plt.tight_layout()
plt.show()

print('\nKey insight: noise strong enough to degrade stealing (std>2) also degrades legitimate users.')
print('There is no free lunch — every defense has a utility cost.')

## 6. Watermark-Based Detection (Preview)

Rather than preventing extraction, watermarking lets you *prove* your model was stolen. We embed a backdoor-style pattern: the victim returns a specific class for a secret trigger input. If the surrogate inherits this behavior, it's forensic evidence.

In [ ]:
def embed_watermark(model, watermark_inputs, watermark_target=0, n_steps=50, lr=1e-3):
    """
    Fine-tune the victim model to output watermark_target on secret trigger inputs.
    The watermark is a secret (trigger, target-class) pair — an ownership proof.
    """
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    wm_inputs = watermark_inputs.to(device)
    wm_labels = torch.full((len(wm_inputs),), watermark_target, dtype=torch.long, device=device)

    for step in range(n_steps):
        opt.zero_grad()
        loss = F.cross_entropy(model(wm_inputs), wm_labels)
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        preds = model(wm_inputs).argmax(1)
    wm_acc = (preds == wm_labels).float().mean().item()
    print(f'Watermark fidelity after fine-tuning: {wm_acc:.3f} (should be ~1.0)')
    return model


def check_watermark(model, watermark_inputs, watermark_target=0):
    """Check if a suspected surrogate reproduces the watermark behavior."""
    model.eval()
    with torch.no_grad():
        preds = model(watermark_inputs.to(device)).argmax(1)
    wm_labels = torch.full((len(preds),), watermark_target, dtype=torch.long, device=device)
    return (preds == wm_labels).float().mean().item()


# Create secret trigger: slightly noisy random images (the 'key')
torch.manual_seed(999)
watermark_inputs = torch.randn(20, 1, 28, 28) * 0.1  # small noise images
watermark_target = 7   # secret: these inputs should always predict class 7

# Embed watermark in victim model
print('Embedding ownership watermark in victim model...')
victim_model = embed_watermark(victim_model, watermark_inputs, watermark_target)

# Re-evaluate victim accuracy (watermark should not degrade it much)
victim_model.eval()
correct = sum(
    (victim_model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    for x, y in test_loader
)
victim_acc_wm = correct / len(test_dataset)
print(f'Victim accuracy after watermarking: {victim_acc_wm:.4f}')

# Now rebuild surrogate via extraction of the watermarked victim
api_wm = VictimAPI(victim_model)
surrogate_wm = knockoff_extraction(api_wm, transfer_images[:5000], n_epochs=5, use_soft_labels=True)
acc_wm, agree_wm = evaluate_surrogate(surrogate_wm, test_loader, victim_model)

# Check if surrogate inherited the watermark
victim_wm_rate = check_watermark(victim_model, watermark_inputs, watermark_target)
surrogate_wm_rate = check_watermark(surrogate_wm, watermark_inputs, watermark_target)
print(f'\nWatermark check on victim:    {victim_wm_rate:.3f} (expected: 1.0)')
print(f'Watermark check on surrogate: {surrogate_wm_rate:.3f} (if >0.5, watermark was stolen too)')
print(f'Surrogate accuracy: {acc_wm:.4f}, agreement: {agree_wm:.4f}')

# A random model should not trigger the watermark
random_model = SurrogateCNN().to(device)
random_wm_rate = check_watermark(random_model, watermark_inputs, watermark_target)
print(f'Random (untrained) model watermark rate: {random_wm_rate:.3f} (expected: ~0.1)')

## Summary

### What We Built

| Component | Key Technique | Result |
|-----------|--------------|--------|
| `VictimAPI` | Black-box wrapper with rate limiting and output noise | Enforced adversary's access restrictions |
| `knockoff_extraction()` | Knowledge distillation on (query, soft-label) pairs | Surrogate ≥90% agreement with 10k queries |
| Soft vs. hard labels | KL divergence loss vs. cross-entropy | Soft labels reach agreement faster per query |
| `adaptive_extraction()` | Uncertainty sampling (entropy-guided active learning) | Same accuracy with fewer queries than random |
| Output perturbation defense | Gaussian noise on logits | Degrades stealing but also hurts legitimate users |
| `embed_watermark()` | Backdoor-style ownership proof | Surrogate inherits watermark through distillation |

### Key Takeaways

- **Soft labels are devastatingly informative.** The probability distribution encodes dark knowledge — inter-class similarities the victim learned — and soft-label extraction consistently outperforms hard-label extraction per query.
- **The surrogate doesn't need to match the victim architecturally.** A smaller CNN can match a larger one's *outputs* without matching its *representations*.
- **Active learning reduces the query budget 2-5x.** Adversaries who are budget-conscious (every API call costs money) can use their current surrogate's uncertainty to guide where to probe next.
- **Output perturbation is a double-edged sword.** Noise strong enough to degrade extraction also degrades legitimate users — there's no free defense.
- **Watermarks transfer through distillation.** If the victim's watermark behavior survives distillation into the surrogate, you have forensic proof of theft without ever having seen the surrogate's weights.

### The Bigger Picture

Model stealing sits at the uncomfortable intersection of ML security and intellectual property law. The attack is asymmetric: extraction is cheap for the adversary but expensive to defend without degrading the legitimate API. The most practically useful defense is probably **watermarking + detection** — you can't prevent the theft, but you can prove it happened in court.

Next: Notebook 13 covers differential privacy fundamentals — the mathematical framework for formally bounding how much information a model leaks about its training data, laying the groundwork for DP-SGD.